In [1]:
import pandas as pd
import sqlite3
# Pandas approach
df_customers = pd.DataFrame({
'customer_id': [1, 2, 3],
'name': ['Jan Kowalski', 'Anna Nowak', 'Piotr Wiśniewski'],
'city': ['Warszawa', 'Kraków', 'Gdańsk'],
'age': [28, 35, 42]
})
# Filtrowanie w Pandas
young_customers = df_customers[df_customers['age'] < 40]
print("Pandas - młodzi klienci:")
print(young_customers)
# SQL approach (SQLite w pamięci)
conn = sqlite3.connect(':memory:')
df_customers.to_sql('customers', conn, index=False)
# To samo zapytanie w SQL
query = """
SELECT *
FROM customers
WHERE age < 40
"""
sql_result = pd.read_sql_query(query, conn)
print("\nSQL - młodzi klienci:")
print(sql_result)
conn.close()

Pandas - młodzi klienci:
   customer_id          name      city  age
0            1  Jan Kowalski  Warszawa   28
1            2    Anna Nowak    Kraków   35

SQL - młodzi klienci:
   customer_id          name      city  age
0            1  Jan Kowalski  Warszawa   28
1            2    Anna Nowak    Kraków   35


In [2]:
import sqlite3
import pandas as pd
# Tworzymy bazę danych (plik na dysku)
conn = sqlite3.connect('sklep.db')
cursor = conn.cursor()
# Tworzymy tabelę klientów
cursor.execute('''
CREATE TABLE IF NOT EXISTS customers (
customer_id INTEGER PRIMARY KEY,
name TEXT NOT NULL,
email TEXT UNIQUE,
city TEXT,
registration_date DATE
)
''')
# Wstawiamy przykładowe dane
customers_data = [
(1, 'Jan Kowalski', 'jan@email.com', 'Warszawa', '2024-01-10'),
(2, 'Anna Nowak', 'anna@email.com', 'Kraków', '2024-01-12'),
(3, 'Piotr Wiśniewski', 'piotr@email.com', 'Gdańsk', '2024-01-15')
]
cursor.executemany('''
INSERT OR REPLACE INTO customers (customer_id, name, email, city, registration_date)
VALUES (?, ?, ?, ?, ?)
''', customers_data)
# Zapisujemy zmiany
conn.commit()
# Odczytujemy dane
df = pd.read_sql_query("SELECT * FROM customers", conn)
print(df)
conn.close()

   customer_id              name            email      city registration_date
0            1      Jan Kowalski    jan@email.com  Warszawa        2024-01-10
1            2        Anna Nowak   anna@email.com    Kraków        2024-01-12
2            3  Piotr Wiśniewski  piotr@email.com    Gdańsk        2024-01-15


In [4]:
import sqlite3
import pandas as pd
# Tworzymy przykładową bazę z produktami
conn = sqlite3.connect(':memory:')
products = pd.DataFrame({
'product_id': [1, 2, 3, 4, 5],
'name': ['Laptop', 'Mysz', 'Klawiatura', 'Monitor', 'Słuchawki'],
'category': ['Elektronika', 'Akcesoria', 'Akcesoria', 'Elektronika', 'Akcesoria'],
'price': [3500, 50, 200, 1200, 150],
'stock': [10, 150, 80, 25, 60]
})
products.to_sql('products', conn, index=False)
# Zapytanie 1: Wszystkie kolumny
query1 = "SELECT * FROM products"
print("Wszystkie produkty:")
print(pd.read_sql_query(query1, conn))
# Zapytanie 2: Wybrane kolumny
query2 = "SELECT name, price FROM products"
print("\nTylko nazwa i cena:")
print(pd.read_sql_query(query2, conn))
# Zapytanie 3: Obliczenia w SELECT
query3 = """
SELECT
name,
price,
price * 1.23 AS price_with_vat,
stock * price AS inventory_value
FROM products
"""
print("\nCena z VAT i wartość zapasów:")
print(pd.read_sql_query(query3, conn))
conn.close()

Wszystkie produkty:
   product_id        name     category  price  stock
0           1      Laptop  Elektronika   3500     10
1           2        Mysz    Akcesoria     50    150
2           3  Klawiatura    Akcesoria    200     80
3           4     Monitor  Elektronika   1200     25
4           5   Słuchawki    Akcesoria    150     60

Tylko nazwa i cena:
         name  price
0      Laptop   3500
1        Mysz     50
2  Klawiatura    200
3     Monitor   1200
4   Słuchawki    150

Cena z VAT i wartość zapasów:
         name  price  price_with_vat  inventory_value
0      Laptop   3500          4305.0            35000
1        Mysz     50            61.5             7500
2  Klawiatura    200           246.0            16000
3     Monitor   1200          1476.0            30000
4   Słuchawki    150           184.5             9000


In [5]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
# Dataset: Titanic (subset)
titanic = pd.DataFrame({
'passenger_id': [1, 2, 3, 4, 5, 6, 7, 8],
'name': ['Braund', 'Cumings', 'Heikkinen', 'Futrelle', 'Allen', 'Moran', 'McCarthy',
'Palsson'],
'age': [22, 38, 26, 35, 35, None, 54, 2],
'sex': ['male', 'female', 'female', 'female', 'male', 'male', 'male', 'male'],
'survived': [0, 1, 1, 1, 0, 0, 0, 0],
'pclass': [3, 1, 3, 1, 3, 3, 1, 3],
'fare': [7.25, 71.28, 7.92, 53.10, 8.05, 8.46, 51.86, 21.08]
})
titanic.to_sql('titanic', conn, index=False)
# Zapytanie 1: Prosty filtr
query1 = """
SELECT name, age, sex, survived
FROM titanic
WHERE survived = 1
"""
print("Ocaleni pasażerowie:")
print(pd.read_sql_query(query1, conn))
# Zapytanie 2: Złożone warunki (AND, OR)
query2 = """
SELECT name, age, sex, pclass, fare
FROM titanic
WHERE (sex = 'female' AND pclass = 1) OR fare > 50
"""
print("\nKobiety z 1 klasy LUB bilet > 50:")
print(pd.read_sql_query(query2, conn))
# Zapytanie 3: Zakres wartości (BETWEEN)
query3 = """
SELECT name, age, fare
FROM titanic
WHERE age BETWEEN 20 AND 40
AND fare < 20
"""
print("\nWiek 20-40 i tani bilet:")
print(pd.read_sql_query(query3, conn))
# Zapytanie 4: Lista wartości (IN)
query4 = """
SELECT name, pclass, fare
FROM titanic
WHERE pclass IN (1, 2)
ORDER BY fare DESC
"""
print("\nKlasa 1 lub 2, sortowane po cenie:")
print(pd.read_sql_query(query4, conn))
conn.close()


Ocaleni pasażerowie:
        name   age     sex  survived
0    Cumings  38.0  female         1
1  Heikkinen  26.0  female         1
2   Futrelle  35.0  female         1

Kobiety z 1 klasy LUB bilet > 50:
       name   age     sex  pclass   fare
0   Cumings  38.0  female       1  71.28
1  Futrelle  35.0  female       1  53.10
2  McCarthy  54.0    male       1  51.86

Wiek 20-40 i tani bilet:
        name   age  fare
0     Braund  22.0  7.25
1  Heikkinen  26.0  7.92
2      Allen  35.0  8.05

Klasa 1 lub 2, sortowane po cenie:
       name  pclass   fare
0   Cumings       1  71.28
1  Futrelle       1  53.10
2  McCarthy       1  51.86


In [6]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
# California Housing (subset dla przykładu)
housing = pd.DataFrame({
'id': range(1, 11),
'median_income': [8.3, 8.3, 7.3, 5.6, 3.8, 4.0, 3.7, 3.5, 3.2, 2.9],
'median_house_value': [452600, 358500, 352100, 341300, 342200, 269700, 299200, 241400,
226700, 261100],
'total_rooms': [880, 7099, 1467, 1274, 1627, 919, 1022, 850, 1665, 1284],
'population': [322, 2401, 496, 558, 565, 413, 372, 341, 845, 341],
'latitude': [37.88, 37.86, 37.85, 37.85, 37.85, 37.84, 37.84, 37.84, 37.84, 37.83],
'longitude': [-122.23, -122.22, -122.24, -122.25, -122.25, -122.25, -122.25, -122.26,
-122.25, -122.27]
})
housing.to_sql('housing', conn, index=False)
# Zapytanie 1: TOP 5 najdroższych domów
query1 = """
SELECT id, median_income, median_house_value
FROM housing
ORDER BY median_house_value DESC
LIMIT 5
"""
print("TOP 5 najdroższych:")
print(pd.read_sql_query(query1, conn))
# Zapytanie 2: Sortowanie po wielu kolumnach
query2 = """
SELECT id, median_income, median_house_value, total_rooms
FROM housing
ORDER BY median_income DESC, total_rooms ASC
LIMIT 7
"""
print("\nSortowane: income DESC, potem rooms ASC:")
print(pd.read_sql_query(query2, conn))
# Zapytanie 3: OFFSET (pomijanie wyników) – paginacja
query3 = """
SELECT id, median_house_value
FROM housing
ORDER BY median_house_value DESC
LIMIT 3 OFFSET 3
"""
print("\nWyniki 4-6 (pomijamy 3 pierwsze):")
print(pd.read_sql_query(query3, conn))
conn.close()


TOP 5 najdroższych:
   id  median_income  median_house_value
0   1            8.3              452600
1   2            8.3              358500
2   3            7.3              352100
3   5            3.8              342200
4   4            5.6              341300

Sortowane: income DESC, potem rooms ASC:
   id  median_income  median_house_value  total_rooms
0   1            8.3              452600          880
1   2            8.3              358500         7099
2   3            7.3              352100         1467
3   4            5.6              341300         1274
4   6            4.0              269700          919
5   5            3.8              342200         1627
6   7            3.7              299200         1022

Wyniki 4-6 (pomijamy 3 pierwsze):
   id  median_house_value
0   5              342200
1   4              341300
2   7              299200


In [9]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
housing = pd.DataFrame({
  'id': range(1, 11),
  'median_income': [8.3, 8.3, 7.3, 5.6, 3.8, 4.0, 3.7, 3.5, 3.2, 2.9],
  'median_house_value': [452600, 358500, 352100, 341300, 342200, 269700, 299200, 241400,226700, 261100],
})
housing.to_sql('housing', conn, index=False)
# Zapytanie: TOP 7 według wartości
query = """
SELECT id, median_income, median_house_value
FROM housing
ORDER BY median_house_value DESC
LIMIT 7
"""
top_houses = pd.read_sql_query(query, conn)
# Tekstowa wizualizacja (1 znak = 10_000 USD)
print("TOP 7 najdroższych domów (1 znak '█' = 10 000 USD):\n")
print(f"{'ID':>3} {'Wartość (USD)':>14} Wykres")
print("-" * 70)
for _, row in top_houses.iterrows():
  bar = '█' * int(row['median_house_value'] / 10_000)
  print(f"{row['id']:>3} {row['median_house_value']:>14,} {bar}")
conn.close()


TOP 7 najdroższych domów (1 znak '█' = 10 000 USD):

 ID  Wartość (USD) Wykres
----------------------------------------------------------------------
1.0      452,600.0 █████████████████████████████████████████████
2.0      358,500.0 ███████████████████████████████████
3.0      352,100.0 ███████████████████████████████████
5.0      342,200.0 ██████████████████████████████████
4.0      341,300.0 ██████████████████████████████████
7.0      299,200.0 █████████████████████████████
6.0      269,700.0 ██████████████████████████


In [10]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
# Tabela klientów
customers = pd.DataFrame({
'customer_id': [1, 2, 3, 4],
'name': ['Jan Kowalski', 'Anna Nowak', 'Piotr Wiśniewski', 'Maria Zając'],
'city': ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
})
# Tabela zamówień
orders = pd.DataFrame({
'order_id': [101, 102, 103, 104, 105],
'customer_id': [1, 2, 1, 3, 1],
'order_date': ['2024-01-10', '2024-01-11', '2024-01-12', '2024-01-13', '2024-01-14'],
'total_amount': [250.0, 120.5, 340.0, 80.0, 190.0]
})
customers.to_sql('customers', conn, index=False)
orders.to_sql('orders', conn, index=False)
# INNER JOIN – tylko klienci którzy złożyli zamówienie
query = """
SELECT
c.customer_id,
c.name,
c.city,
o.order_id,
o.order_date,
o.total_amount
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_date
"""
result = pd.read_sql_query(query, conn)
print("INNER JOIN - klienci z zamówieniami:")
print(result)
# Podsumowanie zamówień na klienta
query2 = """
SELECT
c.name,
COUNT(o.order_id) AS number_of_orders,
SUM(o.total_amount) AS total_spent
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_spent DESC
"""
print("\nPodsumowanie zamówień:")
print(pd.read_sql_query(query2, conn))
conn.close()

INNER JOIN - klienci z zamówieniami:
   customer_id              name      city  order_id  order_date  total_amount
0            1      Jan Kowalski  Warszawa       101  2024-01-10         250.0
1            1      Jan Kowalski  Warszawa       103  2024-01-12         340.0
2            1      Jan Kowalski  Warszawa       105  2024-01-14         190.0
3            2        Anna Nowak    Kraków       102  2024-01-11         120.5
4            3  Piotr Wiśniewski    Gdańsk       104  2024-01-13          80.0

Podsumowanie zamówień:
               name  number_of_orders  total_spent
0      Jan Kowalski                 3        780.0
1        Anna Nowak                 1        120.5
2  Piotr Wiśniewski                 1         80.0


In [12]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
customers = pd.DataFrame({
'customer_id': [1, 2, 3, 4],
'name': ['Jan Kowalski', 'Anna Nowak', 'Piotr Wiśniewski', 'Maria Zając'],
'city': ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
})
orders = pd.DataFrame({
'order_id': [101, 102, 103, 104, 105],
'customer_id': [1, 2, 1, 3, 1],
'order_date': ['2024-01-10', '2024-01-11', '2024-01-12', '2024-01-13', '2024-01-14'],
'total_amount': [250.0, 120.5, 340.0, 80.0, 190.0]
})
customers.to_sql('customers', conn, index=False)
orders.to_sql('orders', conn, index=False)
# LEFT JOIN - wszyscy klienci (nawet bez zamówień)
query = """
SELECT
c.customer_id,
c.name,
c.city,
COUNT(o.order_id) AS number_of_orders,
COALESCE(SUM(o.total_amount), 0) AS total_spent
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.city
ORDER BY total_spent DESC
"""
result = pd.read_sql_query(query, conn)
print("LEFT JOIN - wszyscy klienci:")
print(result)
# Wykrycie klientów bez zamówień
query2 = """
SELECT
c.customer_id,
c.name,
c.city
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
"""
print("\nKlienci BEZ zamówień:")
print(pd.read_sql_query(query2, conn))
conn.close()


LEFT JOIN - wszyscy klienci:
   customer_id              name      city  number_of_orders  total_spent
0            1      Jan Kowalski  Warszawa                 3        780.0
1            2        Anna Nowak    Kraków                 1        120.5
2            3  Piotr Wiśniewski    Gdańsk                 1         80.0
3            4       Maria Zając   Wrocław                 0          0.0

Klienci BEZ zamówień:
   customer_id         name     city
0            4  Maria Zając  Wrocław


In [13]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
# Klienci
customers = pd.DataFrame({
'customer_id': [1, 2, 3],
'name': ['Jan', 'Anna', 'Piotr']
})
# Zamówienia
orders = pd.DataFrame({
'order_id': [101, 102, 103],
'customer_id': [1, 2, 1],
'order_date': ['2024-01-10', '2024-01-11', '2024-01-12']
})
# Pozycje zamówień (order items)
order_items = pd.DataFrame({
'item_id': [1, 2, 3, 4, 5],
'order_id': [101, 101, 102, 103, 103],
'product_name': ['Laptop', 'Mysz', 'Klawiatura', 'Monitor', 'Słuchawki'],
'quantity': [1, 2, 1, 1, 1],
'price': [3500, 50, 200, 1200, 150]
})
customers.to_sql('customers', conn, index=False)
orders.to_sql('orders', conn, index=False)
order_items.to_sql('order_items', conn, index=False)
# JOIN 3 tabel
query = """
SELECT
c.name AS customer_name,
o.order_id,
o.order_date,
oi.product_name,
oi.quantity,
oi.price,
(oi.quantity * oi.price) AS item_total
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
INNER JOIN order_items oi ON o.order_id = oi.order_id
ORDER BY c.name, o.order_id, oi.item_id
"""
result = pd.read_sql_query(query, conn)
print("Szczegóły zamówień (3 tabele):")
print(result)
# Łączna wartość zamówienia dla każdego klienta
query2 = """
SELECT
c.name,
COUNT(DISTINCT o.order_id) AS total_orders,
SUM(oi.quantity * oi.price) AS total_spent
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
INNER JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.name
ORDER BY total_spent DESC
"""
print("\nPodsumowanie wydatków:")
print(pd.read_sql_query(query2, conn))
conn.close()

Szczegóły zamówień (3 tabele):
  customer_name  order_id  order_date product_name  quantity  price  \
0          Anna       102  2024-01-11   Klawiatura         1    200   
1           Jan       101  2024-01-10       Laptop         1   3500   
2           Jan       101  2024-01-10         Mysz         2     50   
3           Jan       103  2024-01-12      Monitor         1   1200   
4           Jan       103  2024-01-12    Słuchawki         1    150   

   item_total  
0         200  
1        3500  
2         100  
3        1200  
4         150  

Podsumowanie wydatków:
   name  total_orders  total_spent
0   Jan             2         4950
1  Anna             1          200


In [14]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
# Tabela sprzedaży
sales = pd.DataFrame({
'sale_id': range(1, 11),
'product': ['Laptop', 'Laptop', 'Mysz', 'Klawiatura', 'Monitor',
'Laptop', 'Mysz', 'Monitor', 'Słuchawki', 'Klawiatura'],
'category': ['Elektronika', 'Elektronika', 'Akcesoria', 'Akcesoria', 'Elektronika',
'Elektronika', 'Akcesoria', 'Elektronika', 'Akcesoria', 'Akcesoria'],
'quantity': [1, 2, 5, 3, 1, 1, 10, 2, 4, 2],
'revenue': [3500, 7000, 250, 600, 1200, 3500, 500, 2400, 600, 400]
})
sales.to_sql('sales', conn, index=False)
# GROUP BY kategoria
query1 = """
SELECT
category,
COUNT(*) AS number_of_sales,
SUM(quantity) AS total_quantity,
SUM(revenue) AS total_revenue,
AVG(revenue) AS avg_revenue,
MIN(revenue) AS min_revenue,
MAX(revenue) AS max_revenue
FROM sales
GROUP BY category
ORDER BY total_revenue DESC
"""
print("Agregacje według kategorii:")
print(pd.read_sql_query(query1, conn))
# GROUP BY produkt
query2 = """
SELECT
product,
COUNT(*) AS times_sold,
SUM(quantity) AS total_units,
SUM(revenue) AS total_revenue
FROM sales
GROUP BY product
ORDER BY total_revenue DESC
LIMIT 5
"""
print("\nTOP 5 produktów:")
print(pd.read_sql_query(query2, conn))
conn.close()

Agregacje według kategorii:
      category  number_of_sales  total_quantity  total_revenue  avg_revenue  \
0  Elektronika                5               7          17600       3520.0   
1    Akcesoria                5              24           2350        470.0   

   min_revenue  max_revenue  
0         1200         7000  
1          250          600  

TOP 5 produktów:
      product  times_sold  total_units  total_revenue
0      Laptop           3            4          14000
1     Monitor           2            3           3600
2  Klawiatura           2            5           1000
3        Mysz           2           15            750
4   Słuchawki           1            4            600


In [15]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
# Dane sprzedażowe
sales = pd.DataFrame({
'sale_id': range(1, 21),
'product': ['Laptop', 'Mysz', 'Klawiatura', 'Monitor', 'Słuchawki'] * 4,
'category': (['Elektronika'] * 2 + ['Akcesoria'] * 3) * 4,
'revenue': [3500, 50, 200, 1200, 150,
3600, 45, 210, 1250, 140,
3400, 55, 190, 1180, 160,
3550, 48, 205, 1220, 155]
})
sales.to_sql('sales', conn, index=False)
# HAVING - tylko produkty z przychdem > 1000
query = """
SELECT
product,
COUNT(*) AS number_of_sales,
SUM(revenue) AS total_revenue,
AVG(revenue) AS avg_revenue
FROM sales
GROUP BY product
HAVING SUM(revenue) > 1000
ORDER BY total_revenue DESC
"""
print("Produkty z przychodem > 1000:")
print(pd.read_sql_query(query, conn))
# HAVING z wieloma warunkami
query2 = """
SELECT
product,
COUNT(*) AS sales_count,
AVG(revenue) AS avg_revenue
FROM sales
GROUP BY product
HAVING COUNT(*) >= 3 AND AVG(revenue) > 500
ORDER BY avg_revenue DESC
"""
print("\n>= 3 sprzedaży i średnia > 500:")
print(pd.read_sql_query(query2, conn))
conn.close()


Produkty z przychodem > 1000:
   product  number_of_sales  total_revenue  avg_revenue
0   Laptop                4          14050       3512.5
1  Monitor                4           4850       1212.5

>= 3 sprzedaży i średnia > 500:
   product  sales_count  avg_revenue
0   Laptop            4       3512.5
1  Monitor            4       1212.5


In [17]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')
# Dane sprzedażowe z datami
sales = pd.DataFrame({
'date': ['2024-01', '2024-01', '2024-01', '2024-02', '2024-02', '2024-02',
'2024-03', '2024-03', '2024-03', '2024-01', '2024-02', '2024-03'],
'category': ['Elektronika', 'Akcesoria', 'Elektronika', 'Elektronika', 'Akcesoria',
'Elektronika',
'Elektronika', 'Akcesoria', 'Elektronika', 'Akcesoria', 'Elektronika',
'Akcesoria'],
'revenue': [3500, 250, 1200, 3600, 300, 1250, 3400, 280, 1180, 200, 1220, 290]
})
sales.to_sql('sales', conn, index=False)
# GROUP BY miesiąc + kategoria
query = """
SELECT
date AS month,
category,
COUNT(*) AS sales_count,
SUM(revenue) AS total_revenue
FROM sales
GROUP BY date, category
ORDER BY date, category
"""
result = pd.read_sql_query(query, conn)
print("Sprzedaż: miesiąc x kategoria (długa forma):")
print(result)
# Pivot w Pandas (L12): miesiące w rzędach, kategorie w kolumnach
pivot_data = result.pivot(index='month', columns='category',
values='total_revenue').fillna(0)
print("\nPivot (miesiąc x kategoria, wartości = przychód):")
print(pivot_data)
# Tekstowa prezentacja (1 znak '█' = 500 PLN)
print("\nPrzychód według miesiąca i kategorii (1 znak = 500 PLN):\n")
for month in pivot_data.index:
  print(f"{month}:")
  for category in pivot_data.columns:
    value = pivot_data.loc[month, category]
    bar = '█' * int(value / 500)
    print(f" {category:<12} {int(value):>6} PLN {bar}")
  print()
conn.close()


Sprzedaż: miesiąc x kategoria (długa forma):
     month     category  sales_count  total_revenue
0  2024-01    Akcesoria            2            450
1  2024-01  Elektronika            2           4700
2  2024-02    Akcesoria            1            300
3  2024-02  Elektronika            3           6070
4  2024-03    Akcesoria            2            570
5  2024-03  Elektronika            2           4580

Pivot (miesiąc x kategoria, wartości = przychód):
category  Akcesoria  Elektronika
month                           
2024-01         450         4700
2024-02         300         6070
2024-03         570         4580

Przychód według miesiąca i kategorii (1 znak = 500 PLN):

2024-01:
 Akcesoria       450 PLN 
 Elektronika    4700 PLN █████████

2024-02:
 Akcesoria       300 PLN 
 Elektronika    6070 PLN ████████████

2024-03:
 Akcesoria       570 PLN █
 Elektronika    4580 PLN █████████



In [19]:
import time
durations = []
for _ in range(5):
  start = time.perf_counter()

# ... zapytanie SQL ...
durations.append(time.perf_counter() - start)
srednia = sum(durations) / len(durations)


In [22]:
import sqlite3
import pandas as pd
import numpy as np
import time
conn = sqlite3.connect(':memory:')

# Tworzymy dużą tabelę (50k wierszy)
np.random.seed(42)
large_data = pd.DataFrame({
  'user_id': range(1, 50001),
  'name': [f'User_{i}' for i in range(1, 50001)],
  'age': np.random.randint(18, 80, 50000),
  'city': np.random.choice(['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław', 'Poznań'], 50000),
  'registration_date': pd.date_range('2020-01-01', periods=50000, freq='15min')
})
large_data.to_sql('users', conn, index=False)
query = "SELECT * FROM users WHERE age > 60 ORDER BY registration_date DESC LIMIT 100"

# === Test 1: BEZ indeksu ===
# Plan wykonania
plan_before = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
print("Plan wykonania BEZ indeksu:")
print(plan_before)

# Pomiar czasu (uruchamiamy kilka razy dla stabilności i uśredniamy)
durations = []
for _ in range(5):
  start = time.perf_counter()
  result_before = pd.read_sql_query(query, conn)
  durations.append(time.perf_counter() - start)
time_before = sum(durations) / len(durations)
print(f"\nŚredni czas BEZ indeksu: {time_before*1000:.2f} ms")

# === Test 2: Z indeksem ===
conn.execute("CREATE INDEX idx_age ON users(age)")
conn.execute("CREATE INDEX idx_registration_date ON users(registration_date)")

# Plan wykonania
plan_after = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
print("\nPlan wykonania Z indeksem:")
print(plan_after)

# Pomiar czasu
durations = []
for _ in range(5):
  start = time.perf_counter()
  result_after = pd.read_sql_query(query, conn)
  durations.append(time.perf_counter() - start)
time_after = sum(durations) / len(durations)
print(f"\nŚredni czas Z indeksem: {time_after*1000:.2f} ms")
print(f"Przyspieszenie: {time_before / time_after:.1f}x")

# Wyświetlenie wyników
print("\nPierwsze 5 wyników zapytania:")
print(result_after.head())

conn.close()

Plan wykonania BEZ indeksu:
   id  parent  notused                        detail
0   4       0        0                    SCAN users
1  21       0        0  USE TEMP B-TREE FOR ORDER BY

Średni czas BEZ indeksu: 15.01 ms

Plan wykonania Z indeksem:
   id  parent  notused                                        detail
0   5       0        0  SCAN users USING INDEX idx_registration_date

Średni czas Z indeksem: 1.05 ms
Przyspieszenie: 14.2x

Pierwsze 5 wyników zapytania:
   user_id        name  age     city    registration_date
0    50000  User_50000   70   Gdańsk  2021-06-04 19:45:00
1    49993  User_49993   72   Kraków  2021-06-04 18:00:00
2    49991  User_49991   72   Gdańsk  2021-06-04 17:30:00
3    49990  User_49990   71  Wrocław  2021-06-04 17:15:00
4    49981  User_49981   72   Poznań  2021-06-04 15:00:00


In [25]:
import sqlite3
import pandas as pd
import numpy as np
import time
conn = sqlite3.connect(':memory:')

# Duże tabele dla realistycznego testu
customers = pd.DataFrame({
'customer_id': range(1, 10001),
'name': [f'Customer_{i}' for i in range(1, 10001)],
'city': ['Warszawa'] * 10000
})

np.random.seed(42)
orders = pd.DataFrame({
'order_id': range(1, 50001),
'customer_id': np.random.randint(1, 10001, 50000),
'amount': np.random.uniform(10, 1000, 50000)
})

customers.to_sql('customers', conn, index=False)
orders.to_sql('orders', conn, index=False)

query = """
SELECT c.name, COUNT(o.order_id) AS order_count, SUM(o.amount) AS total
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
HAVING COUNT(o.order_id) > 5
ORDER BY total DESC
LIMIT 10
"""

# === BEZ indeksu na kluczu obcym ===
plan_before = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
print("Plan JOIN BEZ indeksu na customer_id w orders:")
print(plan_before)

#Pomiar czasu (średnia z 3 powtórzeń)
durations = []
for _ in range(3):
  start = time.perf_counter()
  _ = pd.read_sql_query(query, conn)
  durations.append(time.perf_counter() - start)
time_before = sum(durations) / len(durations)
print(f"\nCzas JOIN BEZ indeksu: {time_before*1000:.2f} ms")

# === Z indeksem na kluczu obcym ===
conn.execute("CREATE INDEX idx_customer_id ON orders(customer_id)")

plan_after = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
print("\nPlan JOIN Z indeksem na customer_id w orders:")
print(plan_after)

durations = []
for _ in range(3):
  start = time.perf_counter()
  result = pd.read_sql_query(query, conn)
  durations.append(time.perf_counter() - start)
time_after = sum(durations) / len(durations)
print(f"\nCzas JOIN Z indeksem: {time_after*1000:.2f} ms")
print(f"Przyspieszenie: {time_before / time_after:.1f}x")

# Wynik końcowy
print("\nTOP 10 klientów:")
print(result)

conn.close()


Plan JOIN BEZ indeksu na customer_id w orders:
   id  parent  notused                                             detail
0   9       0        0                                             SCAN c
1  13       0        0                  BLOOM FILTER ON o (customer_id=?)
2  25       0        0  SEARCH o USING AUTOMATIC COVERING INDEX (custo...
3  32       0        0                       USE TEMP B-TREE FOR GROUP BY
4  85       0        0                       USE TEMP B-TREE FOR ORDER BY

Czas JOIN BEZ indeksu: 60.09 ms

Plan JOIN Z indeksem na customer_id w orders:
   id  parent  notused                                             detail
0  10       0        0                                             SCAN c
1  12       0        0  SEARCH o USING INDEX idx_customer_id (customer...
2  19       0        0                       USE TEMP B-TREE FOR GROUP BY
3  72       0        0                       USE TEMP B-TREE FOR ORDER BY

Czas JOIN Z indeksem: 40.32 ms
Przyspieszenie: 1.5x

TOP 1

In [26]:
import sqlite3
import pandas as pd
import time
conn = sqlite3.connect(':memory:')

# Tabela logów użytkownika
logs = pd.DataFrame({
  'log_id': range(1, 20001),
  'user_id': [i % 100 + 1 for i in range(20000)],
  'action': ['login', 'logout', 'view', 'purchase'] * 5000,
  'timestamp': pd.date_range('2024-01-01', periods=20000, freq='1min')
})
logs.to_sql('user_logs', conn, index=False)

# Zapytanie często używane w aplikacji
query = """
SELECT user_id, action, COUNT(*) AS count
FROM user_logs
WHERE user_id = 42 AND action = 'purchase'
GROUP BY user_id, action
"""

# === BEZ indeksu ===
plan_before = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
print("BEZ indeksu:")
print(plan_before)
durations = []
for _ in range(5):
  start = time.perf_counter()
  _ = pd.read_sql_query(query, conn)
  durations.append(time.perf_counter() - start)
time_before = sum(durations) / len(durations)
print(f"Średni czas BEZ indeksu: {time_before*1000:.3f} ms")

# === Z indeksem złożonym ===
conn.execute("CREATE INDEX idx_user_action ON user_logs(user_id, action)")
plan_after = pd.read_sql_query(f"EXPLAIN QUERY PLAN {query}", conn)
print("\nZ indeksem złożonym (user_id, action):")
print(plan_after)

durations = []
for _ in range(5):
  start = time.perf_counter()
  result = pd.read_sql_query(query, conn)
  durations.append(time.perf_counter() - start)
time_after = sum(durations) / len(durations)
print(f"Średni czas Z indeksem: {time_after*1000:.3f} ms")
print(f"Przyspieszenie: {time_before / time_after:.1f}x")

# Wynik
print("\nWynik zapytania:")
print(result)
conn.close()


BEZ indeksu:
   id  parent  notused          detail
0   6       0        0  SCAN user_logs
Średni czas BEZ indeksu: 1.900 ms

Z indeksem złożonym (user_id, action):
   id  parent  notused                                             detail
0   6       0        0  SEARCH user_logs USING COVERING INDEX idx_user...
Średni czas Z indeksem: 0.620 ms
Przyspieszenie: 3.1x

Wynik zapytania:
Empty DataFrame
Columns: [user_id, action, count]
Index: []


In [34]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')

# Przykładowa tabela z poprawioną długością serii
data = pd.DataFrame({
  'id': range(1, 100001),
  'category': (['A', 'B', 'C'] * 33334)[:100000],
  'value': range(1, 100001)
})
data.to_sql('data', conn, index=False)

# ❌ ZŁE - pobiera WSZYSTKIE dane, filtruje w Pythonie
bad_query = "SELECT * FROM data"
df = pd.read_sql_query(bad_query, conn)
filtered = df[df['category'] == 'A'] # Filtrowanie w pamięci
print(f"ZŁE: Pobrano {len(df)} wierszy, zużyto {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

# ✅ DOBRE - filtruje w bazie
good_query = "SELECT * FROM data WHERE category = 'A'"
df_good = pd.read_sql_query(good_query, conn)
print(f"DOBRE: Pobrano {len(df_good)} wierszy, zużyto {df_good.memory_usage(deep=True).sum() / 1024:.1f} KB")
conn.close()

ZŁE: Pobrano 100000 wierszy, zużyto 6445.4 KB
DOBRE: Pobrano 33334 wierszy, zużyto 2148.6 KB


In [35]:
# ❌ ZŁE - pobiera miliony wierszy niepotrzebnie
bad = "SELECT * FROM huge_table"
# ✅ DOBRE - tylko potrzebne dane
good = "SELECT * FROM huge_table LIMIT 1000"

In [36]:
# ❌ ZŁE - 50 kolumn, z czego używamy 2
bad = "SELECT * FROM users JOIN orders JOIN products ..."
# ✅ DOBRE - tylko potrzebne
good = "SELECT users.name, orders.total FROM users JOIN orders ..."

In [40]:
import sqlite3
import pandas as pd

# Ponowne nawiązanie połączenia i utworzenie przykładowych danych, ponieważ poprzednie połączenie zostało zamknięte
conn = sqlite3.connect(':memory:')
sales_data = pd.DataFrame({
    'category': ['Elektronika', 'Akcesoria', 'Elektronika', 'Akcesoria'],
    'revenue': [3500, 250, 1200, 300]
})
sales_data.to_sql('sales', conn, index=False)

# ❌ ZŁE - transfer 1 GB danych do Pythona
df = pd.read_sql_query("SELECT * FROM sales", conn)
summary = df.groupby('category')['revenue'].sum()

# ✅ DOBRE - 3 wiersze wynikowe
summary_sql = pd.read_sql_query("""
SELECT category, SUM(revenue) AS total
FROM sales
GROUP BY category
""", conn)

print("Podsumowanie z SQL:\n", summary_sql)
conn.close()

Podsumowanie z SQL:
       category  total
0    Akcesoria    550
1  Elektronika   4700


In [43]:
import sqlite3
import pandas as pd
import time
conn = sqlite3.connect(':memory:')

# Duża tabela
large = pd.DataFrame({
'id': range(1, 100001),
'category': ['A', 'B', 'C', 'D', 'E'] * 20000,
'value': range(1, 100001)
})
large.to_sql('large_table', conn, index=False)

# ❌ Podejście nieefektywne: pełny transfer + filtr + agregacja w Pythonie
start = time.perf_counter()
df_all = pd.read_sql_query("SELECT * FROM large_table", conn)
result_bad = df_all[df_all['category'] == 'A']['value'].sum()
time_bad = time.perf_counter() - start
mem_bad = df_all.memory_usage(deep=True).sum() / 1024 # KB
print(f"❌ Nieefektywne: pobrano {len(df_all):,} wierszy, zużyto {mem_bad:.1f} KB, "
f"czas {time_bad*1000:.2f} ms, wynik = {result_bad:,}")

# ✅ Podejście efektywne: agregacja w SQL
start = time.perf_counter()
result_good_df = pd.read_sql_query("""
SELECT SUM(value) AS total
FROM large_table
WHERE category = 'A'
""", conn)
result_good = result_good_df.iloc[0, 0]
time_good = time.perf_counter() - start
mem_good = result_good_df.memory_usage(deep=True).sum() / 1024 # KB
print(f"✅ Efektywne: pobrano 1 wiersz, zużyto {mem_good:.1f} KB, "
f"czas {time_good*1000:.2f} ms, wynik = {result_good:,}")
conn.close()


❌ Nieefektywne: pobrano 100,000 wierszy, zużyto 6445.4 KB, czas 415.25 ms, wynik = 999,970,000
✅ Efektywne: pobrano 1 wiersz, zużyto 0.1 KB, czas 19.07 ms, wynik = 999,970,000


In [46]:
import sqlite3
import pandas as pd

# Wczytanie Titanic dataset
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)

# Tworzenie bazy
conn = sqlite3.connect('titanic.db')
titanic.to_sql('passengers', conn, if_exists='replace', index=False)

# Tworzenie indeksów
conn.execute("CREATE INDEX IF NOT EXISTS idx_survived ON passengers(Survived)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_pclass ON passengers(Pclass)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_sex ON passengers(Sex)")

# Analiza 1: Przeżywalność według płci i klasy
query1 = """
SELECT
Sex,
Pclass,
COUNT(*) AS total_passengers,
SUM(Survived) AS survived,
ROUND(AVG(Survived) * 100, 1) AS survival_rate
FROM passengers
GROUP BY Sex, Pclass
ORDER BY Sex, Pclass
"""
survival_analysis = pd.read_sql_query(query1, conn)
print("Przeżywalność: płeć x klasa")
print(survival_analysis)

# Pivot (L12): wiersze = klasa, kolumny = płeć
pivot = survival_analysis.pivot(index='Pclass', columns='Sex', values='survival_rate')
print("\nPivot (Pclass x Sex, wartości = % przeżywalności):")
print(pivot)

# Tekstowa wizualizacja (1 znak = 2% przeżywalności)
print("\nPrzeżywalność wg klasy i płci (1 znak '█' = 2%):\n")
for pclass in pivot.index:
  print(f"Klasa {pclass}:")
  for sex in pivot.columns:
    rate = pivot.loc[pclass, sex]
    bar = '█' * int(rate / 2)
    print(f" {sex:<7} {rate:>5.1f}% {bar}")
  print()

# Analiza 2: Średni wiek przeżywających vs nie-przeżywających
query2 = """
SELECT
CASE WHEN Survived = 1 THEN 'Ocaleni' ELSE 'Nie-ocaleni' END AS status,
COUNT(*) AS count,
ROUND(AVG(Age), 1) AS avg_age,
MIN(Age) AS min_age,
MAX(Age) AS max_age
FROM passengers
WHERE Age IS NOT NULL
GROUP BY Survived
"""
age_analysis = pd.read_sql_query(query2, conn)
print("Wiek: ocaleni vs nie-ocaleni")
print(age_analysis)

# Analiza 3: TOP porty zaokrętowania
query3 = """
SELECT
CASE
WHEN Embarked = 'C' THEN 'Cherbourg'
WHEN Embarked = 'Q' THEN 'Queenstown'
WHEN Embarked = 'S' THEN 'Southampton'
ELSE 'Unknown'
END AS port,
COUNT(*) AS passengers,
SUM(Survived) AS survived,
ROUND(AVG(Survived) * 100, 1) AS survival_rate,
ROUND(AVG(Fare), 2) AS avg_fare
FROM passengers
WHERE Embarked IS NOT NULL
GROUP BY Embarked
ORDER BY survival_rate DESC
"""
port_analysis = pd.read_sql_query(query3, conn)
print("\nPrzeżywalność według portu:")
print(port_analysis)

conn.close()

Przeżywalność: płeć x klasa
      Sex  Pclass  total_passengers  survived  survival_rate
0  female       1                94        91           96.8
1  female       2                76        70           92.1
2  female       3               144        72           50.0
3    male       1               122        45           36.9
4    male       2               108        17           15.7
5    male       3               347        47           13.5

Pivot (Pclass x Sex, wartości = % przeżywalności):
Sex     female  male
Pclass              
1         96.8  36.9
2         92.1  15.7
3         50.0  13.5

Przeżywalność wg klasy i płci (1 znak '█' = 2%):

Klasa 1:
 female   96.8% ████████████████████████████████████████████████
 male     36.9% ██████████████████

Klasa 2:
 female   92.1% ██████████████████████████████████████████████
 male     15.7% ███████

Klasa 3:
 female   50.0% █████████████████████████
 male     13.5% ██████

Wiek: ocaleni vs nie-ocaleni
        status  count  avg

In [48]:
import sqlite3
import pandas as pd
conn = sqlite3.connect('titanic.db')

# Grupowanie wieku w przedziały
query = """
SELECT
CASE
WHEN Age < 12 THEN '0-11 (dzieci)'
WHEN Age < 20 THEN '12-19 (nastolatki)'
WHEN Age < 40 THEN '20-39 (młodzi)'
WHEN Age < 60 THEN '40-59 (średni wiek)'
ELSE '60+ (seniorzy)'
END AS age_group,
COUNT(*) AS total,
SUM(Survived) AS survived,
ROUND(AVG(Survived) * 100, 1) AS survival_rate
FROM passengers
WHERE Age IS NOT NULL
GROUP BY age_group
ORDER BY
CASE
WHEN age_group LIKE '0-11%' THEN 1
WHEN age_group LIKE '12-19%' THEN 2
WHEN age_group LIKE '20-39%' THEN 3
WHEN age_group LIKE '40-59%' THEN 4
ELSE 5
END
"""

age_groups = pd.read_sql_query(query, conn)
print(age_groups)

# Tekstowa wizualizacja (1 znak '█' = 2% przeżywalności)
print("\nPrzeżywalność wg grupy wiekowej (1 znak '█' = 2%):\n")
print(f"{'Grupa wiekowa':<22} {'N':>4} {'Ocaleni':>7} {'Rate':>6} Wykres")
print("-" * 80)
for _, row in age_groups.iterrows():
  bar = '█' * int(row['survival_rate'] / 2)
  print(f"{row['age_group']:<22} {row['total']:>4} {row['survived']:>7}{row['survival_rate']:>5.1f}% {bar}")

conn.close()

             age_group  total  survived  survival_rate
0        0-11 (dzieci)     68        39           57.4
1   12-19 (nastolatki)     96        40           41.7
2       20-39 (młodzi)    387       150           38.8
3  40-59 (średni wiek)    137        54           39.4
4       60+ (seniorzy)     26         7           26.9

Przeżywalność wg grupy wiekowej (1 znak '█' = 2%):

Grupa wiekowa             N Ocaleni   Rate Wykres
--------------------------------------------------------------------------------
0-11 (dzieci)            68      39 57.4% ████████████████████████████
12-19 (nastolatki)       96      40 41.7% ████████████████████
20-39 (młodzi)          387     150 38.8% ███████████████████
40-59 (średni wiek)     137      54 39.4% ███████████████████
60+ (seniorzy)           26       7 26.9% █████████████


In [52]:
# Zadanie 1. Podstawowy SELECT

import sqlite3
import pandas as pd

# Baza danych i tabela
conn = sqlite3.connect("sklep.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS products")  # pozwala uruchamiać skrypt wielokrotnie
cur.execute("""
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name       TEXT    NOT NULL,
    category   TEXT    NOT NULL,
    price      REAL    NOT NULL,
    stock      INTEGER NOT NULL
)
""")

# 10 przykładowych produktów
products = [
    (1,  "Laptop",          "Elektronika", 3499.99,   15),
    (2,  "Smartfon",        "Elektronika", 2199.00,   40),
    (3,  "Słuchawki",       "Elektronika",  349.50,   80),
    (4,  "Krzesło biurowe", "Meble",        599.00,   25),
    (5,  "Biurko",          "Meble",        899.90,   12),
    (6,  "Lampka LED",      "Dom",           79.99,  150),
    (7,  "Czajnik",         "AGD",          139.00,   60),
    (8,  "Blender",         "AGD",          249.99,   35),
    (9,  "Zeszyt A5",       "Papiernicze",    6.50,  500),
    (10, "Długopis",        "Papiernicze",    3.20, 1200),
]
cur.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", products)
conn.commit()

# Zapytanie 1: wszystkie produkty
query_all = "SELECT * FROM products"
df_all = pd.read_sql_query(query_all, conn)
print("SELECT * FROM products")
print(df_all, end="\n\n")

# --- Zapytanie 2: tylko name i price ---------------------------------------
query_name_price = "SELECT name, price FROM products"
df_name_price = pd.read_sql_query(query_name_price, conn)
print("SELECT name, price FROM products")
print(df_name_price)

conn.close()

SELECT * FROM products
   product_id             name     category    price  stock
0           1           Laptop  Elektronika  3499.99     15
1           2         Smartfon  Elektronika  2199.00     40
2           3        Słuchawki  Elektronika   349.50     80
3           4  Krzesło biurowe        Meble   599.00     25
4           5           Biurko        Meble   899.90     12
5           6       Lampka LED          Dom    79.99    150
6           7          Czajnik          AGD   139.00     60
7           8          Blender          AGD   249.99     35
8           9        Zeszyt A5  Papiernicze     6.50    500
9          10         Długopis  Papiernicze     3.20   1200

SELECT name, price FROM products
              name    price
0           Laptop  3499.99
1         Smartfon  2199.00
2        Słuchawki   349.50
3  Krzesło biurowe   599.00
4           Biurko   899.90
5       Lampka LED    79.99
6          Czajnik   139.00
7          Blender   249.99
8        Zeszyt A5     6.50
9  

In [53]:
# Zadanie 2. WHERE z warunkami
import sqlite3
import pandas as pd

# Baza i tabela products z zadania 1
conn = sqlite3.connect("sklep.db")

# 1. Produkty droższe niż 100 PLN
q1 = "SELECT * FROM products WHERE price > 100"
print("1) " + q1)
print(pd.read_sql_query(q1, conn), end="\n\n")

# 2. Kategoria "Elektronika" LUB cena < 50 PLN
q2 = "SELECT * FROM products WHERE category = 'Elektronika' OR price < 50"
print("2) " + q2)
print(pd.read_sql_query(q2, conn), end="\n\n")

# 3. Zapas (stock) między 10 a 50 (włącznie)
q3 = "SELECT * FROM products WHERE stock BETWEEN 10 AND 50"
print("3) " + q3)
print(pd.read_sql_query(q3, conn))

conn.close()

1) SELECT * FROM products WHERE price > 100
   product_id             name     category    price  stock
0           1           Laptop  Elektronika  3499.99     15
1           2         Smartfon  Elektronika  2199.00     40
2           3        Słuchawki  Elektronika   349.50     80
3           4  Krzesło biurowe        Meble   599.00     25
4           5           Biurko        Meble   899.90     12
5           7          Czajnik          AGD   139.00     60
6           8          Blender          AGD   249.99     35

2) SELECT * FROM products WHERE category = 'Elektronika' OR price < 50
   product_id       name     category    price  stock
0           1     Laptop  Elektronika  3499.99     15
1           2   Smartfon  Elektronika  2199.00     40
2           3  Słuchawki  Elektronika   349.50     80
3           9  Zeszyt A5  Papiernicze     6.50    500
4          10   Długopis  Papiernicze     3.20   1200

3) SELECT * FROM products WHERE stock BETWEEN 10 AND 50
   product_id          

In [54]:
# Zadanie 3. ORDER BY i LIMIT
import sqlite3
import pandas as pd

conn = sqlite3.connect("sklep.db")

# 1. 5 najdroższych produktów
q1 = "SELECT * FROM products ORDER BY price DESC LIMIT 5"
print("1) " + q1)
print(pd.read_sql_query(q1, conn), end="\n\n")

# 2. 5 produktów z najmniejszym zapasem
q2 = "SELECT * FROM products ORDER BY stock ASC LIMIT 5"
print("2) " + q2)
print(pd.read_sql_query(q2, conn), end="\n\n")

# 3. Alfabetycznie po nazwie
# Domyślnie SQLite porównuje teksty bajt po bajcie, więc polskie znaki
# (np. "ł") lądują PO wszystkich literach ASCII, czyli po "z".
q3 = "SELECT name, category, price FROM products ORDER BY name ASC"
print("3a) " + q3 + "   (domyślne porównywanie)")
print(pd.read_sql_query(q3, conn), end="\n\n")

# Własne sortowanie zgodne z polskim alfabetem (bez rozróżniania wielkości liter)
ALFABET = "aąbcćdeęfghijklłmnńoópqrsśtuvwxyzźż"
KLUCZ = {znak: i for i, znak in enumerate(ALFABET)}

def polski_klucz(tekst):
    return [KLUCZ.get(z, len(ALFABET) + ord(z)) for z in tekst.lower()]

def polski_porownaj(a, b):
    ka, kb = polski_klucz(a), polski_klucz(b)
    return (ka > kb) - (ka < kb)

conn.create_collation("POLSKI", polski_porownaj)

q3b = "SELECT name, category, price FROM products ORDER BY name COLLATE POLSKI ASC"
print("3b) " + q3b)
print(pd.read_sql_query(q3b, conn))

conn.close()

1) SELECT * FROM products ORDER BY price DESC LIMIT 5
   product_id             name     category    price  stock
0           1           Laptop  Elektronika  3499.99     15
1           2         Smartfon  Elektronika  2199.00     40
2           5           Biurko        Meble   899.90     12
3           4  Krzesło biurowe        Meble   599.00     25
4           3        Słuchawki  Elektronika   349.50     80

2) SELECT * FROM products ORDER BY stock ASC LIMIT 5
   product_id             name     category    price  stock
0           5           Biurko        Meble   899.90     12
1           1           Laptop  Elektronika  3499.99     15
2           4  Krzesło biurowe        Meble   599.00     25
3           8          Blender          AGD   249.99     35
4           2         Smartfon  Elektronika  2199.00     40

3a) SELECT name, category, price FROM products ORDER BY name ASC   (domyślne porównywanie)
              name     category    price
0           Biurko        Meble   899.9

In [55]:
# Zadanie 4. Podstawowy JOIN
import sqlite3
import pandas as pd

conn = sqlite3.connect("sklep.db")
cur = conn.cursor()

# Tabele customers i orders
cur.execute("DROP TABLE IF EXISTS orders")      # orders najpierw (zależy od customers)
cur.execute("DROP TABLE IF EXISTS customers")

cur.execute("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    city        TEXT NOT NULL
)
""")
cur.execute("""
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount      REAL    NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
)
""")

customers = [
    (1, "Anna Kowalska",     "Kraków"),
    (2, "Jan Nowak",         "Warszawa"),
    (3, "Maria Wiśniewska",  "Gdańsk"),
    (4, "Piotr Zieliński",   "Wrocław"),
    (5, "Ewa Lewandowska",   "Poznań"),
    (6, "Tomasz Kamiński",   "Łódź"),      # celowo bez zamówień
]
orders = [
    (1,  1,  250.00),
    (2,  1,  120.50),
    (3,  2,  899.99),
    (4,  3,   45.00),
    (5,  3,  310.75),
    (6,  2,   75.20),
    (7,  4, 1500.00),
    (8,  5,   60.00),
    (9,  1,  430.40),
    (10, 4,  199.99),
]
cur.executemany("INSERT INTO customers VALUES (?, ?, ?)", customers)
cur.executemany("INSERT INTO orders VALUES (?, ?, ?)", orders)
conn.commit()

# INNER JOIN: nazwa klienta, miasto, kwota; od największej kwoty
query = """
SELECT c.name, c.city, o.amount
FROM customers AS c
INNER JOIN orders AS o ON c.customer_id = o.customer_id
ORDER BY o.amount DESC
"""
print(query)
df = pd.read_sql_query(query, conn)
print(df)

# Kontrola: INNER JOIN pomija klientów bez zamówień
print(f"\nKlientów w tabeli: {len(customers)}, "
      f"różnych klientów w wyniku JOIN: {df['name'].nunique()}, "
      f"wierszy w wyniku: {len(df)}")

conn.close()



SELECT c.name, c.city, o.amount
FROM customers AS c
INNER JOIN orders AS o ON c.customer_id = o.customer_id
ORDER BY o.amount DESC

               name      city   amount
0   Piotr Zieliński   Wrocław  1500.00
1         Jan Nowak  Warszawa   899.99
2     Anna Kowalska    Kraków   430.40
3  Maria Wiśniewska    Gdańsk   310.75
4     Anna Kowalska    Kraków   250.00
5   Piotr Zieliński   Wrocław   199.99
6     Anna Kowalska    Kraków   120.50
7         Jan Nowak  Warszawa    75.20
8   Ewa Lewandowska    Poznań    60.00
9  Maria Wiśniewska    Gdańsk    45.00

Klientów w tabeli: 6, różnych klientów w wyniku JOIN: 5, wierszy w wyniku: 10


In [56]:
# Zadanie 5. LEFT JOIN
import sqlite3
import pandas as pd

# Tabele customers i orders z zadania 4
conn = sqlite3.connect("sklep.db")

# 1. LEFT JOIN: wszyscy klienci, także bez zamówień
q1 = """
SELECT c.customer_id, c.name, c.city, o.order_id, o.amount
FROM customers AS c
LEFT JOIN orders AS o ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_id
"""
print("1) LEFT JOIN (brak zamówienia = NULL, w Pandas: None/NaN)")
print(pd.read_sql_query(q1, conn), end="\n\n")

# 2. COALESCE: NULL w kwocie zamieniamy na 0
q2 = """
SELECT c.name, c.city, COALESCE(o.amount, 0) AS amount
FROM customers AS c
LEFT JOIN orders AS o ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_id
"""
print("2) COALESCE(o.amount, 0)")
print(pd.read_sql_query(q2, conn), end="\n\n")

# 3. Klienci, którzy NIE złożyli zamówienia
q3 = """
SELECT c.customer_id, c.name, c.city
FROM customers AS c
LEFT JOIN orders AS o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
"""
print("3) WHERE o.order_id IS NULL")
print(pd.read_sql_query(q3, conn))

conn.close()

1) LEFT JOIN (brak zamówienia = NULL, w Pandas: None/NaN)
    customer_id              name      city  order_id   amount
0             1     Anna Kowalska    Kraków       1.0   250.00
1             1     Anna Kowalska    Kraków       2.0   120.50
2             1     Anna Kowalska    Kraków       9.0   430.40
3             2         Jan Nowak  Warszawa       3.0   899.99
4             2         Jan Nowak  Warszawa       6.0    75.20
5             3  Maria Wiśniewska    Gdańsk       4.0    45.00
6             3  Maria Wiśniewska    Gdańsk       5.0   310.75
7             4   Piotr Zieliński   Wrocław       7.0  1500.00
8             4   Piotr Zieliński   Wrocław      10.0   199.99
9             5   Ewa Lewandowska    Poznań       8.0    60.00
10            6   Tomasz Kamiński      Łódź       NaN      NaN

2) COALESCE(o.amount, 0)
                name      city   amount
0      Anna Kowalska    Kraków   250.00
1      Anna Kowalska    Kraków   120.50
2      Anna Kowalska    Kraków   430.40


In [57]:
# Zadanie 6. Podstawowy GROUP BY
import sqlite3
import pandas as pd

# Tabela orders (i customers) z zadania 4
conn = sqlite3.connect("sklep.db")

# 1. Liczba zamówień dla każdego klienta
q1 = """
SELECT customer_id, COUNT(*) AS liczba_zamowien
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
print("1) COUNT(*)")
print(pd.read_sql_query(q1, conn), end="\n\n")

# 2. Łączna kwota zamówień dla każdego klienta
q2 = """
SELECT customer_id, SUM(amount) AS laczna_kwota
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
print("2) SUM(amount)")
print(pd.read_sql_query(q2, conn), end="\n\n")

# 3. Średnia wartość zamówienia dla każdego klienta
q3 = """
SELECT customer_id, ROUND(AVG(amount), 2) AS srednia_wartosc
FROM orders
GROUP BY customer_id
ORDER BY customer_id
"""
print("3) AVG(amount)")
print(pd.read_sql_query(q3, conn), end="\n\n")

# Dodatkowo: wszystko w jednym zapytaniu, z nazwami klientów
q4 = """
SELECT c.name,
       COUNT(*)                 AS liczba_zamowien,
       SUM(o.amount)            AS laczna_kwota,
       ROUND(AVG(o.amount), 2)  AS srednia_wartosc
FROM orders AS o
INNER JOIN customers AS c ON c.customer_id = o.customer_id
GROUP BY o.customer_id, c.name
ORDER BY laczna_kwota DESC
"""
print("4) Wszystko razem, z nazwami klientów")
print(pd.read_sql_query(q4, conn))

conn.close()



1) COUNT(*)
   customer_id  liczba_zamowien
0            1                3
1            2                2
2            3                2
3            4                2
4            5                1

2) SUM(amount)
   customer_id  laczna_kwota
0            1        800.90
1            2        975.19
2            3        355.75
3            4       1699.99
4            5         60.00

3) AVG(amount)
   customer_id  srednia_wartosc
0            1           266.97
1            2           487.60
2            3           177.88
3            4           850.00
4            5            60.00

4) Wszystko razem, z nazwami klientów
               name  liczba_zamowien  laczna_kwota  srednia_wartosc
0   Piotr Zieliński                2       1699.99           850.00
1         Jan Nowak                2        975.19           487.60
2     Anna Kowalska                3        800.90           266.97
3  Maria Wiśniewska                2        355.75           177.88
4   Ewa Lewandowska

In [59]:
# Zadanie 7. COUNT i SUM
import sqlite3
import pandas as pd

# Tabela products z zadania 1
conn = sqlite3.connect("sklep.db")

# 1. Łączna liczba produktów
q1 = "SELECT COUNT(*) AS liczba_produktow FROM products"
print("1) COUNT(*)")
print(pd.read_sql_query(q1, conn), end="\n\n")

# 2. Łączna wartość zapasów: suma (stock * price) po wszystkich produktach
q2 = "SELECT ROUND(SUM(stock * price), 2) AS wartosc_zapasow FROM products"
print("2) SUM(stock * price)")
print(pd.read_sql_query(q2, conn), end="\n\n")

# 3. Średnia cena produktów w każdej kategorii
q3 = """
SELECT category,
       COUNT(*)                AS liczba_produktow,
       ROUND(AVG(price), 2)    AS srednia_cena
FROM products
GROUP BY category
ORDER BY srednia_cena DESC
"""
print("3) AVG(price) ... GROUP BY category")
print(pd.read_sql_query(q3, conn))

conn.close()

1) COUNT(*)
   liczba_produktow
0                10

2) SUM(stock * price)
   wartosc_zapasow
0         230371.8

3) AVG(price) ... GROUP BY category
      category  liczba_produktow  srednia_cena
0  Elektronika                 3       2016.16
1        Meble                 2        749.45
2          AGD                 2        194.50
3          Dom                 1         79.99
4  Papiernicze                 2          4.85


In [60]:
# Zadanie 8. Tworzenie indeksu, EXPLAIN QUERY PLAN i pomiar czasu
import sqlite3
import time

import numpy as np
import pandas as pd

# Dane: 10 000 rekordów wygenerowanych NumPy (seed 42)
rng = np.random.default_rng(42)
N = 10_000

df = pd.DataFrame({
    "id":          np.arange(1, N + 1),
    "customer_id": rng.integers(1, 5001, size=N),      # ok. 2 rekordy na klienta
    "category":    rng.choice(["A", "B", "C", "D"], size=N),
    "amount":      np.round(rng.uniform(10, 1000, size=N), 2),
})

conn = sqlite3.connect(":memory:")
df.to_sql("transactions", conn, index=False)           # tworzy tabelę i wstawia dane
print("Wierszy w tabeli:", pd.read_sql_query("SELECT COUNT(*) AS n FROM transactions", conn)["n"][0])

# Zapytanie testowe
SZUKANE_ID = 1234
QUERY = "SELECT * FROM transactions WHERE customer_id = ?"
PARAMS = (SZUKANE_ID,)
POWTORZENIA = 5


def zmierz_czas(conn, query, params, powtorzenia=POWTORZENIA):
    """Średni czas zapytania w sekundach z `powtorzenia` uruchomień."""
    conn.execute(query, params).fetchall()             # rozgrzewka (poza pomiarem)
    czasy = []
    for _ in range(powtorzenia):
        start = time.perf_counter()
        wiersze = conn.execute(query, params).fetchall()   # fetchall wymusza wykonanie
        czasy.append(time.perf_counter() - start)
    return float(np.mean(czasy)), len(wiersze)


def plan_zapytania(conn, query, params):
    """Zwraca tekst planu z EXPLAIN QUERY PLAN."""
    plan = pd.read_sql_query("EXPLAIN QUERY PLAN " + query, conn, params=params)
    return " | ".join(plan["detail"])


# 1. BEZ indeksu
czas_bez, ile = zmierz_czas(conn, QUERY, PARAMS)
plan_bez = plan_zapytania(conn, QUERY, PARAMS)

# 2. Tworzymy indeks
conn.execute("CREATE INDEX idx_transactions_customer ON transactions (customer_id)")
conn.commit()

# 3. Z indeksem
czas_z, ile_z = zmierz_czas(conn, QUERY, PARAMS)
plan_z = plan_zapytania(conn, QUERY, PARAMS)

assert ile == ile_z, "Wyniki z indeksem i bez powinny mieć tyle samo wierszy"

# Porównanie z indeksem i bez niego
przyspieszenie = czas_bez / czas_z

print(f"\nZapytanie: {QUERY}  (customer_id = {SZUKANE_ID}), znaleziono wierszy: {ile}\n")

tabela = pd.DataFrame({
    "wariant":         ["BEZ indeksu", "Z indeksem"],
    "sredni_czas_us":  [round(czas_bez * 1e6, 1), round(czas_z * 1e6, 1)],
    "plan":            [plan_bez, plan_z],
})
pd.set_option("display.max_colwidth", None, "display.width", 140)
print(tabela.to_string(index=False))

print(f"\nPrzyspieszenie: {przyspieszenie:.1f}x")
print(f"Różnica czasu:  {(czas_bez - czas_z) * 1e6:.1f} µs")

# Tekstowe słupki (długość proporcjonalna do czasu)
maks = max(czas_bez, czas_z)
for nazwa, t in [("BEZ indeksu", czas_bez), ("Z indeksem ", czas_z)]:
    dlugosc = max(1, round(40 * t / maks))
    print(f"{nazwa} {'█' * dlugosc} {t * 1e6:.1f} µs")

conn.close()


Wierszy w tabeli: 10000

Zapytanie: SELECT * FROM transactions WHERE customer_id = ?  (customer_id = 1234), znaleziono wierszy: 2

    wariant  sredni_czas_us                                                                      plan
BEZ indeksu           361.0                                                         SCAN transactions
 Z indeksem             4.1 SEARCH transactions USING INDEX idx_transactions_customer (customer_id=?)

Przyspieszenie: 88.1x
Różnica czasu:  356.9 µs
BEZ indeksu ████████████████████████████████████████ 361.0 µs
Z indeksem  █ 4.1 µs
